# BPIC17 — augment_and_sample v2

Three targeted prefix-level oversamplings plus one localized input-feature mask. Sits between the Loader and the Training notebook. Reads `Loader/pkl/BPIC_2017_all_5_train.pkl`, applies the toggled transforms, writes the result to `improved/BPIC_2017_all_5_train_augmented_<TAGS>.pkl`. The filename suffix encodes which transforms were applied so multiple variants can coexist.

## Transforms (each individually toggleable)

| Tag | Name | Filter (prefix-level) | Effect |
|---|---|---|---|
| `A1` | A_Denied gateway oversampling | windows where the gateway event (last encoder position) is `A_Denied` | duplicate the matching prefix-target rows by `A1_FACTOR` (5–10x). Shifts the gateway-conditional distribution at A_Denied toward its true successor `O_Refused` (the only directly-follows successor in BPIC17 — the spec said `A_Refused` but no such activity exists in the encoded dict; `O_Refused` covers 2430/2451 = 99% of A_Denied transitions in the train set). |
| `A2` | A_Create Application + Limit raise oversampling | length-1 prefix (only A_Create Application + 4 EOS-padded suffix events), case-level `ApplicationType == 'Limit raise'`, next event is `A_Concept` | duplicate by `A2_FACTOR` (3–5x). Increases exposure to the rarest of the three branches at this gateway. |
| `A3` | W_Handle leads minority-branch oversampling | gateway is `W_Handle leads`; minority branch identified by which next-event class (loop vs `W_Complete application`) is smaller | factor computed automatically as `majority_count / minority_count` so the two branches end at ≈50/50. |
| `M` | Resource feature mask, localized to A_Create Application | every position in the encoder/decoder window where `concept:name == A_Create Application` | overwrite `org:resource` at those positions with `0` (= padding/unknown token, matches the existing pipeline's null encoding). Shortcut-hypothesis test: prevents the LSTM from using Resource as a proxy for ApplicationType / RequestedAmount at this gateway. |

## Naming convention

Output filename is `BPIC_2017_all_5_train_augmented_<TAGS>.pkl` where `<TAGS>` is the underscore-joined concatenation of the enabled transforms (e.g. `A1_A2_A3_M`). If no transform is enabled the suffix is `NONE` and the file is written for completeness only.

## Scope notes

- Augmentations A1–A3 are **train-only**. Val/test pickles in `Loader/pkl/` are untouched so eval metrics stay comparable across augmentation variants.
- The mask experiment (`M`) **must be applied to val and test as well** — the model is supposed to never see Resource at the A_Create Application position, including at evaluation. When `APPLY_MASK` is on, the notebook also writes `BPIC_2017_all_5_val_M.pkl` and `BPIC_2017_all_5_test_M.pkl` next to the augmented train pickle.
- Prefix layout assumed throughout: window length 96, suffix split 4, so the gateway (= last encoder event) is at column `-5` and its first decoder target is at column `-4`. Padding is left-aligned with value `0`.

## Imports & paths

In [1]:
import importlib
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import torch

# Notebook lives at: src/interpretability/improved_pipeline/henryk/bpic17/improved/augmentation_and_sampling/<this>.ipynb
sys.path.insert(0, '../../../../../..')  # -> src/

import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogDataset

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 17
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_TRAIN = Path('../Loader/pkl/BPIC_2017_all_5_train.pkl').resolve()
INPUT_VAL   = Path('../Loader/pkl/BPIC_2017_all_5_val.pkl').resolve()
INPUT_TEST  = Path('../Loader/pkl/BPIC_2017_all_5_test.pkl').resolve()
OUTPUT_DIR  = Path('..').resolve()
print(f'Train:  {INPUT_TRAIN}')
print(f'Val:    {INPUT_VAL}')
print(f'Test:   {INPUT_TEST}')
print(f'Output: {OUTPUT_DIR}/')

Train:  /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/Loader/pkl/BPIC_2017_all_5_train.pkl
Val:    /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/Loader/pkl/BPIC_2017_all_5_val.pkl
Test:   /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/Loader/pkl/BPIC_2017_all_5_test.pkl
Output: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/


## Toggles & per-transform configuration

Set each `APPLY_*` flag and tune the per-transform constants. Re-run the whole notebook after changing flags — each augmentation cell appends to the working tensors so order matters.

In [2]:
# --- transform toggles ---
APPLY_A1 = False   # A_Denied gateway oversampling
APPLY_A2 = False   # A_Create Application + Limit raise oversampling
APPLY_A3 = False   # W_Handle leads minority-branch oversampling
APPLY_M  = True  # Localized Resource mask at A_Create Application positions

# --- per-transform factors ---
A1_FACTOR = 7         # 5 to 10. Total A1-rows added = A1_FACTOR * |A_Denied gateway windows|.
A2_FACTOR = 4         # 3 to 5
# A3_FACTOR is computed automatically from train-set counts (majority / minority).

# --- prefix layout (must match Loader settings) ---
SUFFIX_SPLIT = 4         # last 4 positions are the decoder target window
GATEWAY_COL  = -SUFFIX_SPLIT - 1   # = -5
NEXT_COL     = -SUFFIX_SPLIT       # = -4
PRIOR_COL    = -SUFFIX_SPLIT - 2   # = -6, used to detect length-1 prefixes (= 0 -> no prior event)

# --- categorical feature indices in tensor_list[0] ---
ACT_CAT       = 0   # concept:name
RESOURCE_CAT  = 2   # org:resource
APP_TYPE_CAT  = 6   # case:ApplicationType (case-level, replicated across all positions in the window)

# --- target labels (resolved from all_categories below; constants here for readability) ---
TARGET_ACT_LABELS = {
    'A_Denied': 'A1 gateway',
    'O_Refused': 'A1 expected successor',
    'A_Create Application': 'A2/M anchor',
    'A_Concept': 'A2 expected successor',
    'W_Handle leads': 'A3 gateway / loop branch',
    'W_Complete application': 'A3 exit branch',
}
TARGET_APPTYPE_LABEL = 'Limit raise'

print(f'A1 (A_Denied oversampling)         : {APPLY_A1}  factor={A1_FACTOR}')
print(f'A2 (A_Create+Limit-raise -> A_Concept): {APPLY_A2}  factor={A2_FACTOR}')
print(f'A3 (W_Handle leads minority)       : {APPLY_A3}  factor=auto')
print(f'M  (Resource mask at A_Create App) : {APPLY_M}')

A1 (A_Denied oversampling)         : False  factor=7
A2 (A_Create+Limit-raise -> A_Concept): False  factor=4
A3 (W_Handle leads minority)       : False  factor=auto
M  (Resource mask at A_Create App) : True


## Load train pickle, resolve label indices, copy tensors

We work on cloned tensors so the original `EventLogDataset` is untouched and can be reused for other variants in the same kernel session.

In [3]:
train_dataset = torch.load(str(INPUT_TRAIN), weights_only=False)
print(f'Loaded train dataset: {len(train_dataset)} windows, window_size={train_dataset.encoder_decoder.window_size}')

cat_tensors = [t.clone() for t in train_dataset.tensor_list[0]]   # list[Tensor (N, W)]
num_tensors = [t.clone() for t in train_dataset.tensor_list[1]]   # list[Tensor (N, W)]
_orig_case_ids = train_dataset.tensor_list[2]
if torch.is_tensor(_orig_case_ids):
    case_ids = _orig_case_ids.clone()
else:
    case_ids = list(_orig_case_ids)

# Resolve label indices from the dataset's all_categories so this notebook stays
# robust even if the encoder reorders categories.
act_dict      = train_dataset.all_categories[0][ACT_CAT][2]
app_type_dict = train_dataset.all_categories[0][APP_TYPE_CAT][2]

missing = [name for name in TARGET_ACT_LABELS if name not in act_dict]
if missing:
    raise KeyError(f'Activity labels missing from concept:name dict: {missing}.\n'
                   f'Available labels: {sorted(act_dict.keys())}')
if TARGET_APPTYPE_LABEL not in app_type_dict:
    raise KeyError(f'{TARGET_APPTYPE_LABEL!r} not in case:ApplicationType dict: {app_type_dict}')

A_DENIED       = act_dict['A_Denied']
O_REFUSED      = act_dict['O_Refused']
A_CREATE_APP   = act_dict['A_Create Application']
A_CONCEPT      = act_dict['A_Concept']
W_HANDLE_LEADS = act_dict['W_Handle leads']
W_COMPLETE_APP = act_dict['W_Complete application']
LIMIT_RAISE    = app_type_dict[TARGET_APPTYPE_LABEL]

print(f'\nResolved indices:')
for label, role in TARGET_ACT_LABELS.items():
    print(f'  {label!r:24s} (concept:name)         = {act_dict[label]:>3d}  [{role}]')
print(f'  {TARGET_APPTYPE_LABEL!r:24s} (case:ApplicationType) = {LIMIT_RAISE:>3d}')

applied_tags = []  # filled in by each augmentation block; drives the output filename

Loaded train dataset: 820824 windows, window_size=96

Resolved indices:
  'A_Denied'               (concept:name)         =   6  [A1 gateway]
  'O_Refused'              (concept:name)         =  16  [A1 expected successor]
  'A_Create Application'   (concept:name)         =   5  [A2/M anchor]
  'A_Concept'              (concept:name)         =   4  [A2 expected successor]
  'W_Handle leads'         (concept:name)         =  24  [A3 gateway / loop branch]
  'W_Complete application' (concept:name)         =  23  [A3 exit branch]
  'Limit raise'            (case:ApplicationType) =   2


## §A1 — A_Denied gateway oversampling

Filter: gateway position (`activity[:, -5]`) equals `A_Denied`. We do **not** further filter by the next event — the goal is to shift the model's gateway-conditional distribution at A_Denied to match the empirical one (which is ~99% O_Refused). Multiplying every A_Denied-gateway window by `A1_FACTOR` keeps that empirical shape but increases its weight.

In [4]:
if APPLY_A1:
    activity = cat_tensors[ACT_CAT]
    a1_mask = (activity[:, GATEWAY_COL] == A_DENIED)
    a1_indices = a1_mask.nonzero(as_tuple=True)[0]
    base_count = a1_indices.numel()
    print(f'A1 base windows (gateway = A_Denied): {base_count}')

    # Successor distribution sanity-check: should be dominated by O_Refused.
    succ_counts = Counter(activity[a1_mask, NEXT_COL].tolist())
    inv_act = {v: k for k, v in act_dict.items()}
    inv_act[0] = '<padding>'
    print('  successor distribution at -4 (top 5):')
    for idx, c in succ_counts.most_common(5):
        print(f'    {c:5d}  {inv_act.get(idx, idx)!r}')

    if base_count == 0:
        print('  -> nothing to oversample, skipping A1')
    else:
        # Append the matching rows A1_FACTOR-1 additional times so the total multiplier is A1_FACTOR.
        # (Each base row is already in the dataset once; we add (A1_FACTOR-1) copies.)
        repeats = max(int(A1_FACTOR) - 1, 1)
        rep_idx = a1_indices.repeat(repeats)
        added = rep_idx.numel()
        print(f'  appending {added} rows (factor={A1_FACTOR}, +{repeats}x copies)')
        cat_tensors = [torch.cat([t, t[rep_idx]]) for t in cat_tensors]
        num_tensors = [torch.cat([t, t[rep_idx]]) for t in num_tensors]
        if torch.is_tensor(case_ids):
            case_ids = torch.cat([case_ids, case_ids[rep_idx]])
        else:
            case_ids = list(case_ids) + [case_ids[i] for i in rep_idx.tolist()]
        applied_tags.append('A1')
    print(f'  dataset size after A1: {cat_tensors[0].shape[0]}')
else:
    print('A1 disabled')

A1 disabled


## §A2 — A_Create Application + Limit raise → A_Concept

Filter combines four conditions on the working tensors:
1. gateway is `A_Create Application` (last encoder position)
2. case-level `ApplicationType == 'Limit raise'` (read at gateway position; this attribute is replicated across all event positions of the window)
3. next event is `A_Concept` (the rare branch we want to amplify)
4. position before the gateway is padding (i.e. this is a length-1 prefix — only the very first event of the case is in the encoder)

Note: A1 may have already appended rows to `cat_tensors`. That's fine — those rows have `gateway == A_Denied`, so they cannot match condition (1) and won't be re-counted here.

In [5]:
if APPLY_A2:
    activity = cat_tensors[ACT_CAT]
    app_type = cat_tensors[APP_TYPE_CAT]
    a2_mask = (
        (activity[:, GATEWAY_COL]    == A_CREATE_APP) &
        (activity[:, NEXT_COL]       == A_CONCEPT) &
        (app_type[:, GATEWAY_COL]    == LIMIT_RAISE) &
        (activity[:, PRIOR_COL]      == 0)
    )
    a2_indices = a2_mask.nonzero(as_tuple=True)[0]
    base_count = a2_indices.numel()
    print(f'A2 base windows (Limit raise, length-1 prefix=A_Create App, next=A_Concept): {base_count}')

    if base_count == 0:
        print('  -> nothing to oversample, skipping A2')
    else:
        repeats = max(int(A2_FACTOR) - 1, 1)
        rep_idx = a2_indices.repeat(repeats)
        added = rep_idx.numel()
        print(f'  appending {added} rows (factor={A2_FACTOR}, +{repeats}x copies)')
        cat_tensors = [torch.cat([t, t[rep_idx]]) for t in cat_tensors]
        num_tensors = [torch.cat([t, t[rep_idx]]) for t in num_tensors]
        if torch.is_tensor(case_ids):
            case_ids = torch.cat([case_ids, case_ids[rep_idx]])
        else:
            case_ids = list(case_ids) + [case_ids[i] for i in rep_idx.tolist()]
        applied_tags.append('A2')
    print(f'  dataset size after A2: {cat_tensors[0].shape[0]}')
else:
    print('A2 disabled')

A2 disabled


## §A3 — W_Handle leads minority-branch oversampling

At `W_Handle leads` there are two outgoing branches: loop back (next event = `W_Handle leads`) or exit (next event = `W_Complete application`). We count both in the train set, identify the minority, and append `(majority_count / minority_count - 1)` extra copies of the minority branch's windows so the two branches end up at ≈50/50.

If a third minor successor exists in the data we ignore it for the parity calculation — the spec only contemplates the two named branches.

In [6]:
if APPLY_A3:
    activity = cat_tensors[ACT_CAT]
    a3_gateway = (activity[:, GATEWAY_COL] == W_HANDLE_LEADS)
    a3_loop_mask = a3_gateway & (activity[:, NEXT_COL] == W_HANDLE_LEADS)
    a3_exit_mask = a3_gateway & (activity[:, NEXT_COL] == W_COMPLETE_APP)
    loop_idx = a3_loop_mask.nonzero(as_tuple=True)[0]
    exit_idx = a3_exit_mask.nonzero(as_tuple=True)[0]
    loop_n = loop_idx.numel(); exit_n = exit_idx.numel()
    print(f'A3 W_Handle leads gateway: loop={loop_n}, exit={exit_n}')

    if loop_n == 0 or exit_n == 0:
        print('  -> at least one branch missing; skipping A3')
    else:
        if loop_n < exit_n:
            minority_idx, minority_name, majority_n = loop_idx, 'loop (W_Handle leads)', exit_n
        else:
            minority_idx, minority_name, majority_n = exit_idx, 'exit (W_Complete application)', loop_n
        minority_n = minority_idx.numel()
        target_factor = majority_n / minority_n     # raw factor to bring minority to majority count
        # Each base row is already present once; we need to add (target_factor - 1) extra copies.
        # Use stochastic rounding so the expected post-aug count matches majority_n exactly.
        extra_per_row_int = int(np.floor(target_factor - 1))
        extra_remainder   = (target_factor - 1) - extra_per_row_int   # in [0, 1)
        rng = np.random.RandomState(SEED)
        bonus_mask = rng.random_sample(size=minority_n) < extra_remainder
        bonus_idx = minority_idx[torch.from_numpy(bonus_mask)]
        repeated = minority_idx.repeat(extra_per_row_int) if extra_per_row_int > 0 else minority_idx[:0]
        rep_idx = torch.cat([repeated, bonus_idx])
        added = rep_idx.numel()
        new_minority_n = minority_n + added
        new_share = new_minority_n / (new_minority_n + majority_n)
        print(f'  minority = {minority_name}, factor={target_factor:.3f}x, appending {added} rows')
        print(f'  post-aug minority share at gateway: {new_share:.3f} (target ~0.5)')
        if added > 0:
            cat_tensors = [torch.cat([t, t[rep_idx]]) for t in cat_tensors]
            num_tensors = [torch.cat([t, t[rep_idx]]) for t in num_tensors]
            if torch.is_tensor(case_ids):
                case_ids = torch.cat([case_ids, case_ids[rep_idx]])
            else:
                case_ids = list(case_ids) + [case_ids[i] for i in rep_idx.tolist()]
            applied_tags.append('A3')
    print(f'  dataset size after A3: {cat_tensors[0].shape[0]}')
else:
    print('A3 disabled')

A3 disabled


## §M — Localized Resource mask at A_Create Application positions

For every position in every window where `concept:name == A_Create Application`, overwrite `org:resource` with `0` (= the pipeline's padding/unknown value). The mask is applied **last**, after all oversampling, so the duplicated A2 rows also get masked.

If `APPLY_M=True`, the mask is also applied to clones of the val and test pickles, written next to the augmented train. Without that, evaluation would still feed the model the un-masked Resource at A_Create Application and the experiment is moot.

In [7]:
def apply_resource_mask(cats):
    """Zero org:resource at every position whose activity is A_Create Application.
    Mutates cats[RESOURCE_CAT] in place. Returns (n_positions_masked, n_windows_touched)."""
    activity = cats[ACT_CAT]
    pos_mask = (activity == A_CREATE_APP)
    cats[RESOURCE_CAT] = torch.where(pos_mask, torch.zeros_like(cats[RESOURCE_CAT]), cats[RESOURCE_CAT])
    return int(pos_mask.sum().item()), int(pos_mask.any(dim=1).sum().item())

if APPLY_M:
    n_pos, n_win = apply_resource_mask(cat_tensors)
    print(f'M (train): masked org:resource at {n_pos} positions across {n_win} windows')
    applied_tags.append('M')
else:
    print('M disabled (org:resource left untouched)')

M (train): masked org:resource at 817306 positions across 817306 windows


## Save augmented train pickle

Filename suffix records which transforms were applied; e.g. `A1_A2_A3_M`.

In [8]:
tags_str = '_'.join(applied_tags) if applied_tags else 'NONE'
train_out = OUTPUT_DIR / f'BPIC_2017_all_5_train_augmented_{tags_str}.pkl'

augmented_train = EventLogDataset(
    tensor_tuple=(tuple(cat_tensors), tuple(num_tensors), case_ids),
    all_categories=train_dataset.all_categories,
    encoder_decoder=train_dataset.encoder_decoder,
)

train_out.parent.mkdir(parents=True, exist_ok=True)
torch.save(augmented_train, str(train_out))
print(f'Saved augmented train pickle to {train_out}')
print(f'  Original windows:  {len(train_dataset)}')
print(f'  Augmented windows: {len(augmented_train)}')
print(f'  Growth: {(len(augmented_train) / max(len(train_dataset), 1) - 1) * 100:+.1f}%')
print(f'  Tags applied: {tags_str}')

Saved augmented train pickle to /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/BPIC_2017_all_5_train_augmented_M.pkl
  Original windows:  820824
  Augmented windows: 820824
  Growth: +0.0%
  Tags applied: M


## (Conditional) Write masked val/test pickles

Only runs when `APPLY_M=True`. The output names mirror the train suffix — i.e. only `M` is appended (A1–A3 are train-only and don't change val/test). If `APPLY_M=False` this cell is a no-op.

In [9]:
if APPLY_M:
    for split_name, split_path in (('val', INPUT_VAL), ('test', INPUT_TEST)):
        split_ds = torch.load(str(split_path), weights_only=False)
        s_cat = [t.clone() for t in split_ds.tensor_list[0]]
        s_num = [t.clone() for t in split_ds.tensor_list[1]]
        s_ids = split_ds.tensor_list[2]
        n_pos, n_win = apply_resource_mask(s_cat)
        masked_split = EventLogDataset(
            tensor_tuple=(tuple(s_cat), tuple(s_num), s_ids),
            all_categories=split_ds.all_categories,
            encoder_decoder=split_ds.encoder_decoder,
        )
        out_path = OUTPUT_DIR / f'BPIC_2017_all_5_{split_name}_M.pkl'
        torch.save(masked_split, str(out_path))
        print(f'Saved masked {split_name} pickle: {out_path}')
        print(f'  masked org:resource at {n_pos} positions across {n_win} windows')
else:
    print('APPLY_M disabled; not writing masked val/test pickles.')

Saved masked val pickle: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/BPIC_2017_all_5_val_M.pkl
  masked org:resource at 190427 positions across 190427 windows
Saved masked test pickle: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/BPIC_2017_all_5_test_M.pkl
  masked org:resource at 251907 positions across 251907 windows


## Quick post-augmentation sanity checks

Re-derive the gateway-conditional counts on the augmented train tensors and compare against the original to confirm A1/A2/A3 actually moved the distribution where they were supposed to. This is just diagnostic output; the pickle is already saved above.

In [10]:
def gateway_stats(activity, label, gateway_idx, app_type=None, app_type_idx=None, prior_must_be_pad=False):
    mask = (activity[:, GATEWAY_COL] == gateway_idx)
    if app_type is not None and app_type_idx is not None:
        mask &= (app_type[:, GATEWAY_COL] == app_type_idx)
    if prior_must_be_pad:
        mask &= (activity[:, PRIOR_COL] == 0)
    if not mask.any():
        print(f'  {label}: 0 windows')
        return
    succ = Counter(activity[mask, NEXT_COL].tolist())
    inv = {v: k for k, v in act_dict.items()}
    inv[0] = '<pad>'
    total = sum(succ.values())
    print(f'  {label}: {total} windows; top successors:')
    for idx, c in succ.most_common(4):
        print(f'    {c:6d} ({c/total:5.1%}) -> {inv.get(idx, idx)!r}')

print('=== ORIGINAL train ===')
orig_act = train_dataset.tensor_list[0][ACT_CAT]
orig_app = train_dataset.tensor_list[0][APP_TYPE_CAT]
gateway_stats(orig_act, 'A1 / A_Denied gateway', A_DENIED)
gateway_stats(orig_act, 'A2 / Limit-raise length-1 A_Create App', A_CREATE_APP, orig_app, LIMIT_RAISE, prior_must_be_pad=True)
gateway_stats(orig_act, 'A3 / W_Handle leads gateway', W_HANDLE_LEADS)

print('\n=== AUGMENTED train ===')
aug_act = cat_tensors[ACT_CAT]
aug_app = cat_tensors[APP_TYPE_CAT]
gateway_stats(aug_act, 'A1 / A_Denied gateway', A_DENIED)
gateway_stats(aug_act, 'A2 / Limit-raise length-1 A_Create App', A_CREATE_APP, aug_app, LIMIT_RAISE, prior_must_be_pad=True)
gateway_stats(aug_act, 'A3 / W_Handle leads gateway', W_HANDLE_LEADS)

if APPLY_M:
    pos_mask = (aug_act == A_CREATE_APP)
    res_at_anchor = cat_tensors[RESOURCE_CAT][pos_mask]
    nonzero = (res_at_anchor != 0).sum().item()
    print(f'\nM check: org:resource non-zero values at A_Create Application positions: {nonzero} '
          f'(should be 0 if mask succeeded)')

=== ORIGINAL train ===
  A1 / A_Denied gateway: 2451 windows; top successors:
      2430 (99.1%) -> 'O_Refused'
         8 ( 0.3%) -> 'W_Complete application'
         5 ( 0.2%) -> 'W_Call after offers'
         5 ( 0.2%) -> 'W_Call incomplete files'
  A2 / Limit-raise length-1 A_Create App: 2198 windows; top successors:
      2198 (100.0%) -> 'A_Concept'
  A3 / W_Handle leads gateway: 30790 windows; top successors:
     17489 (56.8%) -> 'W_Handle leads'
     13288 (43.2%) -> 'W_Complete application'
        13 ( 0.0%) -> 'W_Assess potential fraud'

=== AUGMENTED train ===
  A1 / A_Denied gateway: 2451 windows; top successors:
      2430 (99.1%) -> 'O_Refused'
         8 ( 0.3%) -> 'W_Complete application'
         5 ( 0.2%) -> 'W_Call after offers'
         5 ( 0.2%) -> 'W_Call incomplete files'
  A2 / Limit-raise length-1 A_Create App: 2198 windows; top successors:
      2198 (100.0%) -> 'A_Concept'
  A3 / W_Handle leads gateway: 30790 windows; top successors:
     17489 (56.8%) -> '

## To use the result in training

Open `improved/Training/notebook/full_enc_dec_lstm_gn.ipynb` and point the train file path to the variant you produced, e.g.:

```python
USE_AUGMENTED_TRAIN = True
file_path_train = '../../BPIC_2017_all_5_train_augmented_A1_A2_A3.pkl'   # <- match the suffix you saved here
```

When `M` is part of the suffix you must also redirect val (and test, in any evaluation notebook) to the masked pickles produced above:

```python
file_path_val = '../../BPIC_2017_all_5_val_M.pkl'
# and at evaluation:  ../../BPIC_2017_all_5_test_M.pkl
```

Without that the evaluator would feed the model un-masked Resource at A_Create Application and the masking experiment is invalid.